# Plotting

In [2]:
%load_ext autoreload 
%autoreload 2
# reload modules imported 
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from lib.latex_plots import configure_matplotlib
configure_matplotlib()


In [3]:
DATA = Path('../data')
FIGS = Path('../plots/adri')
FIGS.mkdir(exist_ok=True)
EXTERNAL = DATA/'external'
PROCESSED = DATA/'processed'

In [5]:
# ---------- 1. Cargar ----------
works  = pd.read_parquet(EXTERNAL/'works.parquet')
open_access  = pd.read_parquet(EXTERNAL/'work_open_access.parquet')
papers = pd.read_parquet(EXTERNAL/'papers_cleaned.gencat.parquet')
hosts  = pd.read_parquet(EXTERNAL/'host_orgs.parquet')
fields = pd.read_parquet(EXTERNAL/'fields.parquet')
sources = pd.read_parquet(EXTERNAL/'work_sources.parquet')

In [20]:
joined = pd.merge(left=works, right=open_access, left_on='openalex_id', right_on='work_openalex_id')
joined = pd.merge(left=joined, right=fields, left_on='primary_topic_field_id', right_on='field_id')
joined = pd.merge(left=joined, right=sources, left_on='openalex_id', right_on='work_openalex_id')
joined = pd.merge(left=joined, right=hosts, left_on='openalex_id', right_on='openalex_id')
joined

,openalex_id,doi,apc_paid_value_usd,apc_list_value_usd,awards,funders,primary_topic_field_id,topics_field_ids,referenced_works,work_openalex_id_x,...,field_id,field_name,domain_id,domain_name,work_openalex_id_y,source_id,source_display_name,source_type,host_organization_id,host_organization_name
0,W2052543964,10.1021/ja01036a083,NaN,4500.0,[],[],16,[],[],W2052543964,...,16,Chemistry,3,Physical Sciences,W2052543964,S111155417,Journal of the American Chemical Society,journal,P4310320006,American Chemical Society
1,W2917211058,10.1016/j.patrec.2019.02.029,NaN,2550.0,[],[],17,[],[],W2917211058,...,17,Computer Science,3,Physical Sciences,W2917211058,S151820558,Pattern Recognition Letters,journal,P4310320990,Elsevier BV
2,W2126282133,10.1016/j.eswa.2014.05.004,NaN,3500.0,[],[],17,[],[],W2126282133,...,17,Computer Science,3,Physical Sciences,W2126282133,S13144211,Expert Systems with Applications,journal,P4310320990,Elsevier BV
3,W2736298595,10.1016/j.eswa.2017.07.024,NaN,3500.0,[],[],27,[],[],W2736298595,...,27,Medicine,4,Health Sciences,W2736298595,S13144211,Expert Systems with Applications,journal,P4310320990,Elsevier BV
4,W4282960680,10.1016/j.knosys.2022.109244,NaN,3040.0,[],[],17,[],[],W4282960680,...,17,Computer Science,3,Physical Sciences,W4282960680,S10169007,Knowledge-Based Systems,journal,P4310320990,Elsevier BV
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347605,W4283318319,10.1007/s00520-022-07177-1,4290.0,4290.0,"[PS09/00746, PI09/90453, IT620-13, PS09/00910,...","[F4320321705, F4320321919, F4320326506, F43203...",27,"[27, 27, 27]","[W13860778, W1486248547, W1521227887, W1802038...",W4283318319,...,27,Medicine,4,Health Sciences,W4283318319,S50055314,Supportive Care in Cancer,journal,P4310319900,Springer Science+Business Media
347606,W4206474454,10.1109/tpds.2022.3142314,NaN,2195.0,"[U20A20173, 62125206]",[F4320321001],17,"[17, 22, 17]","[W233789407, W1474786465, W2020279330, W204369...",W4206474454,...,17,Computer Science,3,Physical Sciences,W4206474454,S97130795,IEEE Transactions on Parallel and Distributed ...,journal,P4310319808,Institute of Electrical and Electronics Engineers
347607,W4297053029,10.1007/s13593-022-00827-y,2080.0,2080.0,"[CARBOSTOCK (CGL2011-30300, 2012-2014), LIFE18...","[F4320321505, F4320322649, F4320323481, F43203...",11,"[11, 11, 11]","[W1509791622, W1521831729, W1774521583, W19729...",W4297053029,...,11,Agricultural and Biological Sciences,1,Life Sciences,W4297053029,S17425584,Agronomy for Sustainable Development,journal,P4310319900,Springer Science+Business Media
347608,W4296385646,10.1016/j.aprim.2022.102458,1070.0,1070.0,[],[],33,"[33, 33, 27]","[W1995665553, W2000852645, W2039973247, W20482...",W4296385646,...,33,Social Sciences,2,Social Sciences,W4296385646,S48754089,Atención Primaria,journal,P4310320990,Elsevier BV


In [21]:
mask = ~joined['apc_list_value_usd'].isna()
aggregated = joined[mask].groupby(['source_display_name', 'host_organization_name']).agg(
    apc_list_value_usd_mean=('apc_list_value_usd', 'mean'),
    apc_list_value_usd_std=('apc_list_value_usd', 'std'),
    # apc_paid_value_usd_mean=('apc_paid_value_usd', 'mean'),
    # apc_paid_value_usd_std=('apc_paid_value_usd', 'std'),
    is_open_access_mean=('is_open_access', 'mean'),
    count=('openalex_id', 'count'),
)
mask = ~aggregated['apc_list_value_usd_mean'].isna()
masked = aggregated[mask]

display(masked.sort_values('apc_list_value_usd_mean', ascending=False))
display(masked.sort_values('count', ascending=False))
display(masked[masked['is_open_access_mean'] < 1.])

,,apc_list_value_usd_mean,apc_list_value_usd_std,is_open_access_mean,count
source_display_name,host_organization_name,,,,
Nature Biomedical Engineering,Nature Portfolio,12690.000000,0.000000,0.600000,5
Nature Chemical Biology,Nature Portfolio,12570.769231,324.985207,0.846154,26
Nature Cities,Springer Science+Business Media,12556.666667,230.940108,0.666667,3
Nature Materials,Nature Portfolio,12545.555556,370.449328,0.666667,54
Nature Photonics,Nature Portfolio,12513.076923,436.183918,0.769231,13
...,...,...,...,...,...
Nordic journal of nursing research,SAGE Publishing,0.000000,NaN,1.000000,1
Journal of Traditional and Complementary Medicine,Elsevier BV,0.000000,NaN,1.000000,1
Africa Spectrum,SAGE Publishing,0.000000,NaN,1.000000,1


,,apc_list_value_usd_mean,apc_list_value_usd_std,is_open_access_mean,count
source_display_name,host_organization_name,,,,
PLoS ONE,Public Library of Science,2233.229939,278.950309,1.000000,3614
Scientific Reports,Nature Portfolio,2347.040555,363.455048,1.000000,2737
The Science of The Total Environment,Elsevier BV,3907.029428,374.178662,0.642421,1801
Lecture notes in computer science,Springer Science+Business Media,5392.000000,0.000000,0.316479,1602
Astronomy and Astrophysics,EDP Sciences,2024.179133,600.477516,0.999371,1591
...,...,...,...,...,...
Science and Technology of Welding & Joining,Maney Publishing,4400.000000,NaN,1.000000,1
International Journal of Optics,Hindawi Publishing Corporation,1630.000000,NaN,1.000000,1
Neural Development,BioMed Central,2890.000000,NaN,1.000000,1


,,apc_list_value_usd_mean,apc_list_value_usd_std,is_open_access_mean,count
source_display_name,host_organization_name,,,,
AAPS PharmSciTech,Springer Science+Business Media,3806.250000,360.513027,0.375000,8
ACS Applied Bio Materials,American Chemical Society,4833.333333,240.192231,0.740741,27
ACS Applied Electronic Materials,American Chemical Society,4886.363636,211.957553,0.795455,44
ACS Applied Energy Materials,American Chemical Society,4790.000000,249.284691,0.700000,50
ACS Applied Materials & Interfaces,American Chemical Society,4712.686567,247.507770,0.756219,402
...,...,...,...,...,...
memo - Magazine of European Medical Oncology,Springer Science+Business Media,4690.000000,141.421356,0.500000,2
médecine/sciences,EDP Sciences,1725.800000,243.810787,0.800000,5
physica status solidi (RRL) - Rapid Research Letters,Wiley,4362.727273,165.897011,0.636364,11


In [26]:
aggregated.to_frame()

AttributeError: 'DataFrame' object has no attribute 'to_frame'

In [22]:
import numpy as np
d = aggregated
m = d[d.apc_list_value_usd_mean > 0]

fig, ax = plt.subplots(figsize=(9, 6.5))
sc = ax.scatter(m['count'], m.apc_list_value_usd_mean,
                c=m.is_open_access_mean, cmap='RdYlBu_r',
                s=14, alpha=.65, edgecolors='none', zorder=3)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(0.8, m['count'].max()*1.5)
ax.set_ylim(100, 2e4)

# isolíneas: count * apc = gasto total constante
xs = np.logspace(0, np.log10(m['count'].max()*1.5), 300)
for G in [1e4, 1e5, 1e6, 1e7]:
    ys = G / xs
    ok = (ys >= 100) & (ys <= 2e4)
    ax.plot(xs[ok], ys[ok], '--', color='0.5', lw=.8, zorder=1)
    ax.annotate(f'{G/1e6:.0f} M$', (xs[ok][-1], ys[ok][-1]), fontsize=8,
                color='0.4', xytext=(-30, 4), textcoords='offset points')

for name in ['PLoS ONE', 'Scientific Reports', 'Nature Communications',
             'The Science of The Total Environment',
             'Lecture notes in computer science']:
    r = m[m.source_display_name == name]
    if len(r):
        r = r.iloc[0]
        ax.annotate(name, (r['count'], r.apc_list_value_usd_mean), fontsize=8,
                    xytext=(6, 6), textcoords='offset points')

ax.set_xlabel('Nº de artículos publicados (count)')
ax.set_ylabel('APC de lista medio (USD)')
ax.set_title('Volumen vs precio — las discontinuas son gasto total constante')
ax.grid(alpha=.2, which='both')
cb = fig.colorbar(sc, ax=ax, pad=.01); cb.set_label('Fracción de artículos OA')
plt.tight_layout()

AttributeError: 'DataFrame' object has no attribute 'source_display_name'

Error in callback <function _draw_all_if_interactive at 0xf2cdad4d7d80> (for post_execute), with arguments args (),kwargs {}:


RuntimeError: latex was not able to process the following string:
b'0 M$'

Here is the full command invocation and its output:

latex -interaction=nonstopmode -halt-on-error -no-shell-escape file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) (preloaded format=latex)
entering extended mode
(./file.tex
LaTeX2e <2025-11-01>
L3 programming layer <2026-03-20>
(/usr/share/texlive/texmf-dist/tex/latex/base/fix-cm.sty
(/usr/share/texlive/texmf-dist/tex/latex/base/ts1enc.def))
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2025/01/22 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/type1cm/type1cm.sty)
(/usr/share/texlive/texmf-dist/tex/latex/base/inputenc.sty)
(/usr/share/texlive/texmf-dist/tex/latex/geometry/geometry.sty
(/usr/share/texlive/texmf-dist/tex/latex/graphics/keyval.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifvtex.sty
(/usr/sha

RuntimeError: latex was not able to process the following string:
b'0 M$'

Here is the full command invocation and its output:

latex -interaction=nonstopmode -halt-on-error -no-shell-escape file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) (preloaded format=latex)
entering extended mode
(./file.tex
LaTeX2e <2025-11-01>
L3 programming layer <2026-03-20>
(/usr/share/texlive/texmf-dist/tex/latex/base/fix-cm.sty
(/usr/share/texlive/texmf-dist/tex/latex/base/ts1enc.def))
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2025/01/22 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/type1cm/type1cm.sty)
(/usr/share/texlive/texmf-dist/tex/latex/base/inputenc.sty)
(/usr/share/texlive/texmf-dist/tex/latex/geometry/geometry.sty
(/usr/share/texlive/texmf-dist/tex/latex/graphics/keyval.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifvtex.sty
(/usr/sha

<Figure size 900x650 with 1 Axes>